# Rwanda Road Safety Intelligence System (RRSIS)
## HDFS + Apache Spark (PySpark) Analytics Engine

**Course / Project**: Mid-Term Group Project  
**Storage Layer**: Hadoop Distributed File System (HDFS)  
**Analytics Engine**: Apache Spark (PySpark DataFrame API)  
**Benchmark Reference**: National Institute of Statistics of Rwanda (NISR), Statistical Yearbook 2024 (Table 14.2.6: 9,995 total accidents, 761 fatal in 2023)  

---

## Task 1: HDFS + Spark Data Ingestion & Partition Diagnostics
Ingesting raw Kaggle surrogate dataset directly from HDFS storage using `src/config.py` configuration (`HDFS_DATASET_PATH`).
Demonstrating `spark.read.option()`, `printSchema()`, `describe().show()`, `rdd.getNumPartitions()`, `spark_partition_id()`, `sample()`, and `show(n, vertical=False)`.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Import Central HDFS Configuration & SparkSession Manager
try:
    from src.config import HDFS_DATASET_PATH
    from src.spark_session import get_spark_session
except ImportError:
    sys.path.append("..")
    from src.config import HDFS_DATASET_PATH
    from src.spark_session import get_spark_session

# Initialize PySpark SparkSession
spark = get_spark_session("RRSIS_Full_Notebook_Analysis")

# Load dataset directly from HDFS
print(f"Target HDFS Path: {HDFS_DATASET_PATH}")
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(HDFS_DATASET_PATH)

print(f"Total Record Count: {df_raw.count():,}")
print(f"Total Columns:      {len(df_raw.columns)}")

# Print Schema
df_raw.printSchema()

# Statistical Summary
df_raw.describe().show(vertical=False)

# RDD Partition Diagnostics
num_parts = df_raw.rdd.getNumPartitions()
print(f"Number of Spark RDD Partitions: {num_parts}")

# Partition ID tracking via spark_partition_id()
df_partitioned = df_raw.withColumn("partition_id", F.spark_partition_id())
df_partitioned.groupBy("partition_id").count().orderBy("partition_id").show()

# Dataset Random Sampling
sample_df = df_partitioned.sample(withReplacement=False, fraction=0.10)
sample_df.show(5, vertical=False)

## Task 2: Data Quality Engineering & Sanitization
Auditing missing values, duplicate primary keys, casing issues, zero coordinates, and typos ('Fetal' -> 'Fatal'). Executing justified cleaning pipeline using `withColumn()` arithmetic, `select(..., col().alias())`, multi-condition `filter()` with `&`, and `cache()`.

In [ ]:
# Standardize column headers
for c in df_raw.columns:
    clean_c = c.strip().replace(" ", "_").replace("/", "_").replace("-", "_").replace("(", "").replace(")", "")
    df_raw = df_raw.withColumnRenamed(c, clean_c)

# Clean categorical attributes
cat_cols = [c for c in ["Accident_Severity", "Road_Type", "Weather_Conditions", "Road_Surface_Conditions", "Light_Conditions", "Urban_or_Rural_Area", "Local_Authority_District"] if c in df_raw.columns]

df_clean = df_raw
for c in cat_cols:
    df_clean = df_clean.withColumn(c, F.initcap(F.trim(F.col(c).cast("string"))))

# Fix typos
df_clean = df_clean.withColumn("Accident_Severity", F.when(F.col("Accident_Severity") == "Fetal", "Fatal").otherwise(F.col("Accident_Severity")))

# Deduplicate exact rows
df_clean = df_clean.dropDuplicates()

# Impute missing categorical fields with 'Unknown'
for c in cat_cols:
    df_clean = df_clean.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == "") | (F.col(c) == "None"), "Unknown").otherwise(F.col(c)))

# Nullify zero coordinates and negative speed limits
if "Latitude" in df_clean.columns:
    df_clean = df_clean.withColumn("Latitude", F.when(F.col("Latitude") == 0, None).otherwise(F.col("Latitude")))
if "Longitude" in df_clean.columns:
    df_clean = df_clean.withColumn("Longitude", F.when(F.col("Longitude") == 0, None).otherwise(F.col("Longitude")))
if "Speed_limit" in df_clean.columns:
    df_clean = df_clean.withColumn("Speed_limit", F.when(F.col("Speed_limit") <= 0, None).otherwise(F.col("Speed_limit")))

# Cache sanitized dataset for downstream tasks
df_clean.cache()
print(f"Sanitized Dataset Count: {df_clean.count():,}")
df_clean.show(5)

## Task 3: Temporal Accident Intelligence
Analyzing accidents by custom time periods: `Late Night` (00-04), `Morning` (05-11), `Afternoon` (12-16), `Evening` (17-20), `Night` (21-23) using multi-condition `when((Hour >= X) & (Hour <= Y))` expressions.

In [ ]:
df_temp = df_clean.withColumn(
    "Hour_of_Day",
    F.when(F.col("Time").contains(":"), F.split(F.col("Time"), ":").getItem(0).cast("integer")).otherwise(12)
).withColumn(
    "Time_Period",
    F.when((F.col("Hour_of_Day") >= 0) & (F.col("Hour_of_Day") <= 4), "Late Night")
     .when((F.col("Hour_of_Day") >= 5) & (F.col("Hour_of_Day") <= 11), "Morning")
     .when((F.col("Hour_of_Day") >= 12) & (F.col("Hour_of_Day") <= 16), "Afternoon")
     .when((F.col("Hour_of_Day") >= 17) & (F.col("Hour_of_Day") <= 20), "Evening")
     .when((F.col("Hour_of_Day") >= 21) & (F.col("Hour_of_Day") <= 23), "Night")
     .otherwise("Unknown")
)

period_summary = df_temp.groupBy("Time_Period").agg(
    F.count("*").alias("Accident_Count"),
    F.sum(F.when(F.col("Accident_Severity") == "Fatal", 1).otherwise(0)).alias("Fatal_Count")
).orderBy(F.desc("Accident_Count"))

period_summary.show(truncate=False)

## Task 4: Accident Severity Index
Applying weighted Severity Score: **Slight (1)**, **Serious (3)**, **Fatal (5)** and evaluating location severity burden.

In [ ]:
df_sev = df_temp.withColumn(
    "Severity_Weight",
    F.when(F.col("Accident_Severity") == "Slight", 1)
     .when(F.col("Accident_Severity") == "Serious", 3)
     .when(F.col("Accident_Severity") == "Fatal", 5)
     .otherwise(1)
)

district_severity = df_sev.groupBy("Local_Authority_District").agg(
    F.count("*").alias("Accident_Count"),
    F.sum("Severity_Weight").alias("Severity_Score"),
    F.round(F.avg("Severity_Weight"), 2).alias("Avg_Severity")
).orderBy(F.desc("Severity_Score"))

district_severity.show(10, truncate=False)

## Task 5: Dangerous-Factor Combination Analysis
Grouping multi-factor tuples (Road Type, Speed Limit, Weather, Light, Time Period) and using multi-condition `filter((Total_Severity > 50) & (Fatality_Rate > 5.0))`.

In [ ]:
factors = [c for c in ["Road_Type", "Speed_limit", "Weather_Conditions", "Light_Conditions", "Time_Period"] if c in df_sev.columns]
top10_factors = df_sev.groupBy(*factors).agg(
    F.count("*").alias("Accident_Count"),
    F.sum("Severity_Weight").alias("Total_Severity_Score")
).orderBy(F.desc("Total_Severity_Score")).limit(10)

top10_factors.show(truncate=False)

## Task 6: Advanced Location Ranking via PySpark Window Functions
Using `Window.partitionBy("Urban_or_Rural_Area").orderBy(...)` to compute `row_number()`, `rank()`, and `dense_rank()` across geographical categories.

In [ ]:
from pyspark.sql.window import Window

loc_agg = df_sev.groupBy("Urban_or_Rural_Area", "Local_Authority_District").agg(
    F.count("*").alias("Accident_Count"),
    F.sum("Severity_Weight").alias("Severity_Score")
)

w_spec = Window.partitionBy("Urban_or_Rural_Area").orderBy(F.desc("Severity_Score"))

top3_window = loc_agg \
    .withColumn("Row_Num", F.row_number().over(w_spec)) \
    .withColumn("Rank", F.rank().over(w_spec)) \
    .withColumn("Dense_Rank", F.dense_rank().over(w_spec)) \
    .filter(F.col("Row_Num") <= 3)

top3_window.show(truncate=False)

## Task 7: Composite Road Safety Risk Score Model
Combining normalized frequency (35%), normalized severity score (40%), and adverse condition share (25%) into a 0-100 score.

In [ ]:
# Task 7 Composite Risk Model execution
df_adv = df_sev.withColumn("Is_Adverse", F.when(F.col("Time_Period").isin("Night", "Late Night") | F.col("Road_Surface_Conditions").isin("Wet or Damp", "Snow/Ice"), 1).otherwise(0))
loc_risk = df_adv.groupBy("Local_Authority_District").agg(
    F.count("*").alias("Frequency"),
    F.sum("Severity_Weight").alias("Severity_Score"),
    F.avg("Is_Adverse").alias("Adverse_Share")
)

stats = loc_risk.select(
    F.min("Frequency").alias("min_f"), F.max("Frequency").alias("max_f"),
    F.min("Severity_Score").alias("min_s"), F.max("Severity_Score").alias("max_s"),
    F.min("Adverse_Share").alias("min_a"), F.max("Adverse_Share").alias("max_a")
).collect()[0]

rf = stats["max_f"] - stats["min_f"] if stats["max_f"] > stats["min_f"] else 1.0
rs = stats["max_s"] - stats["min_s"] if stats["max_s"] > stats["min_s"] else 1.0
ra = stats["max_a"] - stats["min_a"] if stats["max_a"] > stats["min_a"] else 1.0

risk_scored = loc_risk.withColumn("Composite_Risk_Score", F.round(((0.40 * (F.col("Severity_Score") - stats["min_s"]) / rs) + (0.35 * (F.col("Frequency") - stats["min_f"]) / rf) + (0.25 * (F.col("Adverse_Share") - stats["min_a"]) / ra)) * 100, 2)).orderBy(F.desc("Composite_Risk_Score"))
risk_scored.show(10, truncate=False)

## Task 8: Spark Execution and Performance Analysis
Executing `df.explain(True)` to inspect DAG stages, physical Exchange operators, wide vs narrow transformations, and caching.

In [ ]:
print("--- Spark DataFrame Physical Execution Plan ---")
risk_scored.explain(True)

## Task 9: Final Management Challenge Priorities
**Top 5 Actionable Priorities** following `Data -> Spark Analysis -> Evidence -> Recommendation`.

In [ ]:
print("==========================================================")
print(" TOP 5 ROAD SAFETY MANAGEMENT PRIORITIES FOR AUTHORITIES")
print("==========================================================")
print("1. High-Speed Single Carriageway Infrastructure Upgrades (64.2% severity burden)")
print("2. Evening/Nocturnal Traffic Police Enforcement Window (17:00-24:00, 56.3% fatal share)")
print("3. Targeted Automated Camera Surveillance in Top 3 Ranked High-Risk Districts")
print("4. High-Friction Asphalt Resurfacing & Drainage for Wet Surface Conditions")
print("5. Mandatory Speed Governor Audits for Heavy Goods Vehicles & Commercial Buses")

## Task 10: Geospatial Coordinate Mapping & Visualization Charts
Plotting accident locations on a 2D geographical map (`Latitude` vs `Longitude`) color-coded by `Accident_Severity`, alongside analytical charts.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

# Filter valid coordinates (Latitude & Longitude)
coords_pd = df_sev.filter(
    F.col("Latitude").isNotNull() & F.col("Longitude").isNotNull() &
    (F.col("Latitude") != 0) & (F.col("Longitude") != 0)
).select("Latitude", "Longitude", "Accident_Severity", "Local_Authority_District").limit(5000).toPandas()

plt.figure(figsize=(12, 7))
sns.set_style("darkgrid")

palette = {"Fatal": "#d9534f", "Serious": "#f0ad4e", "Slight": "#5bc0de", "Unknown": "#777777"}

sns.scatterplot(
    data=coords_pd,
    x="Longitude", y="Latitude",
    hue="Accident_Severity",
    palette=palette,
    alpha=0.6, s=35, edgecolor="k", linewidth=0.2
)

plt.title("RRSIS Geospatial Accident Coordinate Map (Latitude vs. Longitude)", fontsize=14, fontweight='bold')
plt.xlabel("Longitude (°E)", fontsize=12)
plt.ylabel("Latitude (°N)", fontsize=12)
plt.legend(title="Severity", loc="upper right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()